# 05 · RQ4: telephone robustness and cost-optimal deployment

1. Scores the paired telephone copies of every test clip and tests clean versus telephone accuracy (McNemar).
2. Chooses decision thresholds on the **validation** subset that minimise expected cost per 1,000 utterances at 1% and 5% prevalence, with a ±50% sensitivity analysis.
3. Saves the deployment bundle used by `python -m src.predict`.

H₀: accuracy is equal on clean and telephone audio. H₁: it is lower on telephone audio.

In [ ]:
# --- Setup: works on Google Colab and locally -------------------------------
import os, sys
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ.setdefault("VS_DATA_ROOT", "/content/drive/MyDrive/audio-detection-human-vs-ai")
    if not os.path.exists("/content/audio-detection-human-vs-ai"):
        !git clone -q https://github.com/abhilash-u/audio-detection-human-vs-ai /content/audio-detection-human-vs-ai
    %cd /content/audio-detection-human-vs-ai
    !pip install -q -r requirements.txt
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import numpy as np, pandas as pd
from src import config as C
C.ensure_dirs()
print("Data folder:", C.DATA_ROOT)

In [ ]:
import joblib, torch
from src import evaluation as E, models as M
from src.features import FEATURES
sample = pd.read_csv(C.META_DIR / "split_assignments.csv")
X = pd.read_csv(C.FEAT_DIR / "features_clean.csv"); L = np.load(C.FEAT_DIR / "logmel_clean.npy")
Xp = pd.read_csv(C.FEAT_DIR / "features_telephone.csv"); Lp = np.load(C.FEAT_DIR / "logmel_telephone.npy")
df = sample.merge(X, on="clip_id"); y = df.label.values
test_ids = Xp.clip_id.values
clean_rows = df.set_index("clip_id").index.get_indexer(test_ids)
y_test = y[clean_rows]
va = np.where(df.split == "validation")[0]
state = pd.read_pickle(C.MODEL_DIR / "rq2_state.pkl")
ck = torch.load(C.MODEL_DIR / "CNN.pt", weights_only=False)
cnn = M.build_cnn(); cnn.load_state_dict(ck["state_dict"])
def score(name, feats, logmels):
    if name == "CNN":
        return M.predict_cnn(cnn, logmels, ck["mean"], ck["std"])
    return joblib.load(C.MODEL_DIR / f"{name}.joblib")["model"].predict_proba(feats[FEATURES])[:, 1]
ku = pd.read_csv(C.REPORT_DIR / "rq3_known_vs_unseen.csv")
best = ku[ku.systems == "unseen"].sort_values("eer").model.iloc[0]
print("Model carried forward:", best)

## Clean versus telephone (paired)

In [ ]:
s_clean = score(best, df.iloc[clean_rows], L[clean_rows])
s_phone = score(best, Xp, Lp)
thr = state["thresholds"][best]
comp = pd.DataFrame([{"channel": "clean", **E.classification_report(y_test, s_clean, thr)},
                     {"channel": "telephone", **E.classification_report(y_test, s_phone, thr)}])
mc = E.mcnemar((s_clean >= thr).astype(int) == y_test, (s_phone >= thr).astype(int) == y_test)
print(f"Clean vs telephone: b={mc['b']} c={mc['c']} p={mc['p_value']:.4f} ({mc['method']})")
comp.to_csv(C.REPORT_DIR / "rq4_clean_vs_telephone.csv", index=False); comp.round(4)

## Cost-optimal thresholds (chosen on validation, applied once to test)
Illustrative costs: missed fraud ₹50,000, false alarm ₹200, manual review ₹100.

In [ ]:
s_val = score(best, df.iloc[va], L[va])
rows = []
for pi in C.PREVALENCES:
    opt = E.cost_optimal_threshold(y[va], s_val, pi)
    t = E.classification_report(y_test, s_phone, opt["threshold"])
    rows.append({"prevalence": pi, "threshold": opt["threshold"], "val_expected_cost": opt["expected_cost"],
                 "test_tpr": t["recall_tpr"], "test_fpr": t["fpr"],
                 "test_expected_cost": E.expected_cost(t["recall_tpr"], t["fpr"], pi),
                 "test_precision_at_pi": E.precision_at_prevalence(t["recall_tpr"], t["fpr"], pi),
                 "cost_no_screening": opt["cost_no_screening"]})
cost = pd.DataFrame(rows); cost.to_csv(C.REPORT_DIR / "rq4_cost_thresholds.csv", index=False); cost.round(4)

## Sensitivity analysis (each cost ±50%)

In [ ]:
sens = []
for pi in C.PREVALENCES:
    for key in ["c_fn", "c_fp", "c_review"]:
        base = {"c_fn": C.COST_FN, "c_fp": C.COST_FP, "c_review": C.COST_REVIEW}
        for f in (0.5, 1.5):
            costs = {**base, key: base[key] * f}
            opt = E.cost_optimal_threshold(y[va], s_val, pi, **costs)
            sens.append({"prevalence": pi, "varied": key, "factor": f, "threshold": opt["threshold"], "expected_cost": opt["expected_cost"]})
sens = pd.DataFrame(sens); sens.to_csv(C.REPORT_DIR / "rq4_sensitivity.csv", index=False); sens.round(4)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(6, 4))
for pi in C.PREVALENCES:
    cc = E.cost_curve(y[va], s_val, pi); ax.plot(cc.threshold.clip(0, 1), cc.expected_cost, label=f"π = {pi:.0%}")
ax.set_xlabel("Decision threshold"); ax.set_ylabel("Expected cost per 1,000 utterances (₹)"); ax.legend()
ax.set_title("RQ4: expected cost by threshold (validation)"); plt.tight_layout(); plt.savefig(C.REPORT_DIR / "rq4_cost_curve.png", dpi=200); plt.show()

## Risk bands and deployment bundle
The **review** threshold is the cost-optimal threshold at 5% prevalence (more sensitive); the **high** threshold is the cost-optimal threshold at 1% prevalence, or the review threshold plus a margin if they coincide.

In [ ]:
t_review = float(cost.loc[cost.prevalence == 0.05, "threshold"].iloc[0])
t_high = float(max(cost.loc[cost.prevalence == 0.01, "threshold"].iloc[0], min(t_review + 0.1, 0.99)))
if best == "CNN":
    bundle = {"kind": "cnn", "state_dict": ck["state_dict"], "mean": ck["mean"], "std": ck["std"]}
else:
    bundle = {"kind": "classical", "model": joblib.load(C.MODEL_DIR / f"{best}.joblib")["model"]}
bundle.update({"name": best, "thresholds": {"review": t_review, "high": t_high}})
joblib.dump(bundle, C.MODEL_DIR / "deployment.joblib")
bands = pd.cut(s_phone, [-np.inf, t_review, t_high, np.inf], labels=["Low", "Review", "High"], right=False)
pd.crosstab(bands, pd.Series(y_test, name="label").map({0: "human", 1: "synthetic"}))

In [ ]:
from src.predict import score_file, load_bundle
b = load_bundle()
score_file(C.PHONE_DIR / f"{test_ids[0]}.wav", b)